# Transparent Agentic Workflow: Google Drive Inventory

**Author:** Alejandro Reynoso  
**Purpose:** Learn how ChatGPT Work moves from a connected plugin and a governing skill to a concrete task and an evidence-backed report.

This notebook makes four layers visible:

| Layer | Meaning | Visible output |
|---|---|---|
| A. Plugin | Authenticated connection to Google Drive | Connection and query log |
| B. Skill | Rules governing how Drive should be searched and interpreted | Decision policy and agent trace |
| C. Task | Inventory files created in the last 48 hours | Structured file table |
| D. Report | Validated summary for a human reader | CSV and HTML downloads |

> **Transparency boundary:** ChatGPT Work plugins and skills run in the Work environment. Colab does not directly execute those internal components. This notebook creates an explicit, reproducible analogue using Google's Drive API, while documenting the same decisions and controls.

## Agentic architecture

```text
USER OBJECTIVE
    ↓
AGENT: interprets scope and time window
    ↓
SKILL POLICY: creation time is primary; modification time is separate
    ↓
PLUGIN / CONNECTOR: authenticated Google Drive access
    ↓
EVIDENCE: file metadata returned by Drive
    ↓
VALIDATION: deduplication, time checks, type classification
    ↓
REPORT: detailed inventory + executive summary + audit trace
```

The **agent** is the orchestrator. The **skill** is the operating policy. The **plugin** supplies authorized capabilities. The **task** defines the desired transformation. The **report** communicates the result and retains evidence.

In [1]:
# 1. Configuration: every consequential task parameter is visible here.
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import json, html, pandas as pd
from IPython.display import display, Markdown

CONFIG = {
    'task_name': 'Inventory of recently added Google Drive files',
    'lookback_hours': 48,
    'timezone': 'America/Mexico_City',
    'include_trashed': False,
    'primary_time_field': 'createdTime',
    'secondary_time_field': 'modifiedTime',
    'output_stem': 'drive_inventory_last_48_hours',
}

now_local = datetime.now(ZoneInfo(CONFIG['timezone']))
start_local = now_local - timedelta(hours=CONFIG['lookback_hours'])
start_utc = start_local.astimezone(timezone.utc)
end_utc = now_local.astimezone(timezone.utc)

print(json.dumps(CONFIG, indent=2))
print(f'Window: {start_local:%Y-%m-%d %H:%M %Z} → {now_local:%Y-%m-%d %H:%M %Z}')

{
  "task_name": "Inventory of recently added Google Drive files",
  "lookback_hours": 48,
  "timezone": "America/Mexico_City",
  "include_trashed": false,
  "primary_time_field": "createdTime",
  "secondary_time_field": "modifiedTime",
  "output_stem": "drive_inventory_last_48_hours"
}
Window: 2026-07-20 06:28 CST → 2026-07-22 06:28 CST


## A. Plugin / connector layer

The Work plugin performs authenticated Drive operations without exposing credentials. In this Colab analogue, Google OAuth provides the same bounded access. The notebook requests read-only metadata access and never deletes, moves, or edits Drive files.

In [2]:
# 2. Authenticate and construct a read-only Google Drive service.
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

credentials, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive.metadata.readonly']
)
drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
print('Connected to Google Drive with metadata read-only scope.')

Connected to Google Drive with metadata read-only scope.


## B. Skill layer: explicit operating policy

The agent follows these rules:

1. Interpret **added** as `createdTime`, not merely `modifiedTime`.
2. Use a rolling 48-hour window in Mexico City time, converted to UTC for the API.
3. Exclude trashed files.
4. Request all pages of results, not only the first page.
5. Preserve file IDs and links as evidence.
6. Classify MIME types consistently and validate counts before reporting.
7. Record actions and decisions in a machine-readable trace.

In [3]:
# 3. Agent trace: observable reasoning at the decision/action level.
TRACE = []
def trace(stage, role, decision, evidence=None):
    TRACE.append({
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'stage': stage, 'agent_role': role, 'decision': decision,
        'evidence': evidence or ''
    })

trace('interpret', 'Task interpreter',
      "Treat 'added' as Drive creation time", CONFIG['primary_time_field'])
trace('scope', 'Temporal planner',
      f"Use a rolling {CONFIG['lookback_hours']}-hour interval",
      f'{start_utc.isoformat()} to {end_utc.isoformat()}')
trace('safety', 'Permission guardian',
      'Use metadata-only read access; make no Drive mutations')
display(pd.DataFrame(TRACE))

,timestamp_utc,stage,agent_role,decision,evidence
0,2026-07-22T12:30:38.556333+00:00,interpret,Task interpreter,Treat 'added' as Drive creation time,createdTime
1,2026-07-22T12:30:38.556422+00:00,scope,Temporal planner,Use a rolling 48-hour interval,2026-07-20T12:28:57.187041+00:00 to 2026-07-22...
2,2026-07-22T12:30:38.556470+00:00,safety,Permission guardian,Use metadata-only read access; make no Drive m...,


## C. Task execution: retrieve and structure the evidence

In [4]:
# 4. Execute the transparent paginated Drive query.
start_rfc3339 = start_utc.strftime('%Y-%m-%dT%H:%M:%SZ')
end_rfc3339 = end_utc.strftime('%Y-%m-%dT%H:%M:%SZ')
query = (f"createdTime >= '{start_rfc3339}' and "
         f"createdTime <= '{end_rfc3339}' and trashed = false")
fields = ('nextPageToken, files(id,name,mimeType,createdTime,modifiedTime,'
          'size,webViewLink,owners(displayName,emailAddress),parents)')

files, page_token, pages = [], None, 0
while True:
    response = drive.files().list(
        q=query, spaces='drive', fields=fields, pageSize=1000,
        pageToken=page_token, orderBy='createdTime desc',
        supportsAllDrives=True, includeItemsFromAllDrives=True,
    ).execute()
    pages += 1
    files.extend(response.get('files', []))
    page_token = response.get('nextPageToken')
    if not page_token:
        break

trace('retrieve', 'Drive retrieval agent', 'Executed paginated metadata query',
      f'{len(files)} files across {pages} API page(s); query={query}')
print(f'Retrieved {len(files):,} files across {pages} page(s).')

Retrieved 462 files across 2 page(s).


In [5]:
# 5. Normalize metadata and classify files.
MIME_LABELS = {
    'application/pdf': 'PDF',
    'application/vnd.openxmlformats-officedocument.wordprocessingml.document': 'Word',
    'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet': 'Excel',
    'application/vnd.openxmlformats-officedocument.presentationml.presentation': 'PowerPoint',
    'application/vnd.google-apps.document': 'Google Doc',
    'application/vnd.google-apps.spreadsheet': 'Google Sheet',
    'application/vnd.google-apps.presentation': 'Google Slides',
    'application/vnd.google-apps.folder': 'Folder',
    'text/markdown': 'Markdown', 'text/plain': 'Plain text',
}
def label_type(mime):
    if mime in MIME_LABELS: return MIME_LABELS[mime]
    if mime.startswith('image/'): return 'Image'
    if mime.startswith('video/'): return 'Video'
    return 'Other'

rows = []
for f in files:
    owners = '; '.join(o.get('displayName') or o.get('emailAddress','')
                       for o in f.get('owners', []))
    rows.append({
        'name': f.get('name'), 'type': label_type(f.get('mimeType','')),
        'mime_type': f.get('mimeType'), 'created_time_utc': f.get('createdTime'),
        'modified_time_utc': f.get('modifiedTime'),
        'size_bytes': int(f['size']) if f.get('size') else None,
        'owner': owners, 'file_id': f.get('id'),
        'drive_link': f.get('webViewLink'),
    })
df = pd.DataFrame(rows)
if not df.empty:
    for c in ['created_time_utc','modified_time_utc']:
        df[c] = pd.to_datetime(df[c], utc=True)
        df[c.replace('_utc','_local')] = df[c].dt.tz_convert(CONFIG['timezone'])
    df = df.sort_values('created_time_utc', ascending=False).reset_index(drop=True)
display(df.head(20))

,name,type,mime_type,created_time_utc,modified_time_utc,size_bytes,owner,file_id,drive_link,created_time_local,modified_time_local
0,Transparent_Agentic_Drive_Inventory.ipynb,Other,application/json,2026-07-22 12:28:13.307000+00:00,2026-07-22 12:30:28.837000+00:00,14393.0,Alejandro Reynoso del Valle,1XHe0C64i-1VVwTfipzMKs7Pq2n912jzc,https://drive.google.com/file/d/1XHe0C64i-1VVw...,2026-07-22 06:28:13.307000-06:00,2026-07-22 06:30:28.837000-06:00
1,THE ESSENTIAL AUTONOMOUS WORKFLOWS,Folder,application/vnd.google-apps.folder,2026-07-21 22:43:26.103000+00:00,2026-07-22 12:27:21.768000+00:00,NaN,Alejandro Reynoso del Valle,1sUfWE8xkKWbH3TQ1YN9Sjx91gS2piI4X,https://drive.google.com/drive/folders/1sUfWE8...,2026-07-21 16:43:26.103000-06:00,2026-07-22 06:27:21.768000-06:00
2,A_Hands_On_Implementation_of_a_Governed_Tax_Pl...,PDF,application/pdf,2026-07-21 18:48:42.680000+00:00,2026-07-21 18:45:20+00:00,725895.0,Alejandro Reynoso del Valle,13RHdGfYiJ7kPhuGDO4km1zI-ckA_rQlJ,https://drive.google.com/file/d/13RHdGfYiJ7kPh...,2026-07-21 12:48:42.680000-06:00,2026-07-21 12:45:20-06:00
3,A_Hands_On_Implementation_of_a_Governed_Tax_Pl...,Word,application/vnd.openxmlformats-officedocument....,2026-07-21 18:48:20.115000+00:00,2026-07-21 18:45:52+00:00,348005.0,Alejandro Reynoso del Valle,1dmd-bwvCBFQ8ax8WB6SHx0Govg85UpTH,https://docs.google.com/document/d/1dmd-bwvCBF...,2026-07-21 12:48:20.115000-06:00,2026-07-21 12:45:52-06:00
4,Tax_Planning_ExoBrain_Steps_0_to_10_Complete_D...,Other,application/zip,2026-07-21 17:11:31.566000+00:00,2026-07-21 17:11:31.566000+00:00,5491813.0,Alejandro Reynoso del Valle,1jwKAQescqxO9BwI3F3Mt_tcz9LONW-LG,https://drive.google.com/file/d/1jwKAQescqxO9B...,2026-07-21 11:11:31.566000-06:00,2026-07-21 11:11:31.566000-06:00
5,Tax_Planning_ExoBrain_All_Colab_Notebooks_Step...,Other,application/zip,2026-07-21 17:11:27.349000+00:00,2026-07-21 17:11:27.349000+00:00,196563.0,Alejandro Reynoso del Valle,1eAzZ7MUu8MIjFz-tkoH3SoaJ3ZtwtIfF,https://drive.google.com/file/d/1eAzZ7MUu8MIjF...,2026-07-21 11:11:27.349000-06:00,2026-07-21 11:11:27.349000-06:00
6,Tax_Planning_ExoBrain_Current_Vault_Step_10.zip,Other,application/zip,2026-07-21 17:11:24.423000+00:00,2026-07-21 17:11:24.423000+00:00,190109.0,Alejandro Reynoso del Valle,1d4Bc9k9ssNt4w6MAFiCO7DJM51-Hyd3O,https://drive.google.com/file/d/1d4Bc9k9ssNt4w...,2026-07-21 11:11:24.423000-06:00,2026-07-21 11:11:24.423000-06:00
7,Tax_Planning_ExoBrain_Step_10.zip,Other,application/zip,2026-07-21 17:11:21.901000+00:00,2026-07-21 17:11:21.901000+00:00,813937.0,Alejandro Reynoso del Valle,1k8jZUlGSh-s6i62ia1FXDpYqsuOcfcJH,https://drive.google.com/file/d/1k8jZUlGSh-s6i...,2026-07-21 11:11:21.901000-06:00,2026-07-21 11:11:21.901000-06:00
8,Tax_Planning_ExoBrain_Step_9.zip,Other,application/zip,2026-07-21 17:11:18.604000+00:00,2026-07-21 17:11:18.604000+00:00,793476.0,Alejandro Reynoso del Valle,1O2f2iVp8jfBcexjP1hvzIUYRM1Y_1G35,https://drive.google.com/file/d/1O2f2iVp8jfBce...,2026-07-21 11:11:18.604000-06:00,2026-07-21 11:11:18.604000-06:00
9,Tax_Planning_ExoBrain_Step_8.zip,Other,application/zip,2026-07-21 17:11:14.065000+00:00,2026-07-21 17:11:14.065000+00:00,774348.0,Alejandro Reynoso del Valle,1E9-d2vWWBDLC2KeipuqISq9-abhC2GCz,https://drive.google.com/file/d/1E9-d2vWWBDLC2...,2026-07-21 11:11:14.065000-06:00,2026-07-21 11:11:14.065000-06:00


In [6]:
# 6. Validation gate: the report is generated only after these checks.
checks = {
    'unique_file_ids': df.empty or not df['file_id'].duplicated().any(),
    'all_created_after_start': df.empty or bool((df['created_time_utc'] >= start_utc).all()),
    'all_created_before_end': df.empty or bool((df['created_time_utc'] <= end_utc).all()),
    'no_missing_names': df.empty or bool(df['name'].notna().all()),
}
validation = pd.DataFrame([{'check': k, 'passed': v} for k,v in checks.items()])
display(validation)
if not all(checks.values()):
    raise ValueError('Validation failed. Inspect the table before producing a report.')
trace('validate', 'Quality-control agent', 'All validation gates passed', str(checks))

,check,passed
0,unique_file_ids,True
1,all_created_after_start,True
2,all_created_before_end,True
3,no_missing_names,True


## D. Report generation

In [7]:
# 7. Create the human report, detailed CSV, and agent audit trace.
summary = (df.groupby('type').size().sort_values(ascending=False)
           .rename('files_added').reset_index()) if not df.empty else pd.DataFrame(columns=['type','files_added'])
display(Markdown(f"### Inventory result: **{len(df):,} files added**"))
display(summary)

csv_path = Path(f"{CONFIG['output_stem']}.csv")
html_path = Path(f"{CONFIG['output_stem']}.html")
trace_path = Path(f"{CONFIG['output_stem']}_agent_trace.json")
df.to_csv(csv_path, index=False)
trace('report', 'Reporting agent', 'Generated inventory and executive report',
      f'{csv_path.name}; {html_path.name}; {trace_path.name}')
trace_path.write_text(json.dumps(TRACE, indent=2), encoding='utf-8')

report_html = f'''<!doctype html><html><head><meta charset="utf-8"><title>Drive Inventory</title>
<style>body{{font-family:Arial,sans-serif;max-width:1200px;margin:40px auto;color:#17243a}}
h1{{color:#8b1e3f}} table{{border-collapse:collapse;width:100%;font-size:13px}}
th,td{{border:1px solid #d9dee7;padding:7px;text-align:left}}th{{background:#eef2f7}}
.meta{{background:#f7f8fa;padding:14px;border-left:5px solid #8b1e3f}}</style></head><body>
<h1>Google Drive: Files Added in the Last {CONFIG['lookback_hours']} Hours</h1>
<div class="meta"><b>Task:</b> {html.escape(CONFIG['task_name'])}<br>
<b>Window:</b> {start_local:%Y-%m-%d %H:%M %Z} to {now_local:%Y-%m-%d %H:%M %Z}<br>
<b>Definition:</b> Added = Drive creation time<br><b>Total:</b> {len(df):,}</div>
<h2>Summary by type</h2>{summary.to_html(index=False)}
<h2>Detailed inventory</h2>{df.to_html(index=False, escape=True)}
<h2>Agent trace</h2>{pd.DataFrame(TRACE).to_html(index=False, escape=True)}
</body></html>'''
html_path.write_text(report_html, encoding='utf-8')
print('Created:', csv_path, html_path, trace_path, sep='\n- ')

### Inventory result: **462 files added**

,type,files_added
0,Markdown,219
1,Other,156
2,Folder,66
3,PDF,11
4,Word,4
5,Plain text,2
6,Image,2
7,Excel,1
8,PowerPoint,1


Created:
- drive_inventory_last_48_hours.csv
- drive_inventory_last_48_hours.html
- drive_inventory_last_48_hours_agent_trace.json


In [8]:
# 8. Download the deliverables from Colab.
from google.colab import files as colab_files
for artifact in [csv_path, html_path, trace_path]:
    colab_files.download(str(artifact))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What this exercise teaches

- A plugin is not the agent; it is an authorized capability.
- A skill is not the report; it is the policy governing tool use and interpretation.
- The agent translates an objective into parameters, invokes capabilities, checks evidence, and prepares a result.
- Transparency does not require exposing private credentials or hidden chain-of-thought. It requires observable inputs, policies, actions, evidence, validation, and outputs.
- Reproducibility comes from parameterized code, complete pagination, explicit time semantics, and retained audit artifacts.